In [16]:
!git clone https://github.com/IDEA-Research/Grounded-SAM-2
%cd Grounded-SAM-2


Cloning into 'Grounded-SAM-2'...
remote: Enumerating objects: 2065, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 2065 (delta 11), reused 5 (delta 5), pack-reused 2046 (from 2)
Receiving objects: 100% (2065/2065), 230.08 MiB | 15.74 MiB/s, done.
Resolving deltas: 100% (769/769), done.
/content/Grounded-SAM-2/Grounded-SAM-2


In [17]:
# Cell 2: Install PyTorch (Colab usually already has this, but ensure correct version)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [15]:
# Cell 3: Set CUDA_HOME (critical for Grounding DINO compilation)
import os
os.environ["CUDA_HOME"] = "/usr/local/cuda"

# Install SAM 2
!pip install -e . -q

# Install Grounding DINO (this compiles the CUDA op — takes ~3-5 min)
!pip install --no-build-isolation -e grounding_dino -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 17.4 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done


In [18]:
# Cell 4: Download model checkpoints
!cd checkpoints && bash download_ckpts.sh
!cd gdino_checkpoints && bash download_ckpts.sh

--2026-06-11 16:49:43--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.9.168.62, 65.9.168.4, 65.9.168.52, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.9.168.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 156008466 (149M) [application/vnd.snesdev-page-table]
Saving to: ‘sam2.1_hiera_tiny.pt’

sam2.1_hiera_tiny.p 100%[===================>] 148.78M   260MB/s    in 0.6s    

2026-06-11 16:49:43 (260 MB/s) - ‘sam2.1_hiera_tiny.pt’ saved [156008466/156008466]

--2026-06-11 16:49:43--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.9.168.62, 65.9.168.4, 65.9.168.52, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.9.168.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 184416285 (176M) [application/vnd.sn

In [19]:
!pip install supervision -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 7.3 MB/s eta 0:00:00


In [20]:
# Cell 5: Run the HuggingFace demo (easiest)
!python grounded_sam2_hf_model_demo.py

preprocessor_config.json: 100% 457/457 [00:00<00:00, 1.83MB/s]
config.json: 100% 1.64k/1.64k [00:00<00:00, 5.17MB/s]
tokenizer_config.json: 100% 1.24k/1.24k [00:00<00:00, 3.33MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 44.1MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 135MB/s]
added_tokens.json: 100% 82.0/82.0 [00:00<00:00, 547kB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 638kB/s]
model.safetensors: 100% 689M/689M [00:04<00:00, 153MB/s]
Loading weights: 100% 978/978 [00:00<00:00, 4689.95it/s]


In [21]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import to_rgba
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor



SAM2_CHECKPOINT  = "./checkpoints/sam2.1_hiera_large.pt"
SAM2_MODEL_CFG   = "configs/sam2.1/sam2.1_hiera_l.yaml"
GDINO_MODEL_ID   = "IDEA-Research/grounding-dino-tiny"
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
BOX_THRESHOLD    = 0.35
TEXT_THRESHOLD   = 0.25

# ── Load models ───────────────────────────────────────────────────────────────
print("Loading Grounding DINO...")
processor = AutoProcessor.from_pretrained(GDINO_MODEL_ID)
gdino     = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_MODEL_ID).to(DEVICE)

print("Loading SAM 2...")
sam2       = build_sam2(SAM2_MODEL_CFG, SAM2_CHECKPOINT, device=DEVICE)
predictor  = SAM2ImagePredictor(sam2)



/content/Grounded-SAM-2/Grounded-SAM-2/sam2/modeling/sam/transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


Loading Grounding DINO...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/978 [00:00<?, ?it/s]

Loading SAM 2...


In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
!find /content/drive -name "novel.zip" 2>/dev/null


/content/drive/MyDrive/novel.zip


In [10]:
import os
for root, dirs, files in os.walk("/content/novel"):
    level = root.replace("/content/novel", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:3]:  # show first 3 files in each folder
        print(f"  {indent}{f}")

novel/
  __MACOSX/
    ._novel
    novel/
      ._000055_novel02.jpg
      ._000030_novel02.jpg
      ._000010_novel02.jpg
  novel/
    000115_novel01.jpg
    000030_novel02.jpg
    000000_novel00.jpg


In [ ]:
import os, glob
!unzip -q "/content/drive/MyDrive/novel.zip" -d "/content/novel"


# ── CONFIG ────────────────────────────────────────────────────────────────────
IMAGE_FOLDER = "/content/novel/novel"   # folder containing your photos
TEXT_PROMPT  = "hands."
OUTPUT_FOLDER = "/content/seg_results"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
# ──────────────────────────────────────────────────────────────────────────────

image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, "*.jpg")) +
                     glob.glob(os.path.join(IMAGE_FOLDER, "*.jpeg")) +
                     glob.glob(os.path.join(IMAGE_FOLDER, "*.png")))

print(f"Found {len(image_paths)} images in {IMAGE_FOLDER}")

for idx, IMAGE_PATH in enumerate(image_paths):
    print(f"\n[{idx+1}/{len(image_paths)}] {os.path.basename(IMAGE_PATH)}")

    image_pil = Image.open(IMAGE_PATH).convert("RGB")
    image_np  = np.array(image_pil)

    # Step 1: Grounding DINO → boxes
    inputs = processor(images=image_pil, text=TEXT_PROMPT, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = gdino(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,
        target_sizes=[image_pil.size[::-1]]
    )[0]

    boxes  = results["boxes"].cpu().numpy()
    labels = results["labels"]
    scores = results["scores"].cpu().numpy()
    print(f"  Detected {len(boxes)} object(s): {labels}")

    # Step 2: SAM 2 → masks
    predictor.set_image(image_np)
    if len(boxes) > 0:
        masks, _, _ = predictor.predict(
            point_coords=None,
            point_labels=None,
            box=boxes,
            multimask_output=False,
        )
        if masks.ndim == 4:
            masks = masks.squeeze(1)
    else:
        masks = []

    # Step 3: Visualize + save
    COLORS = plt.cm.get_cmap("tab10").colors

    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(image_np)

    for i, (mask, label, score, box) in enumerate(zip(masks, labels, scores, boxes)):
        color = COLORS[i % len(COLORS)]

        colored_mask = np.zeros((*mask.shape, 4), dtype=np.float32)
        colored_mask[mask > 0] = [*color[:3], 0.65]
        ax.imshow(colored_mask)

        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)

        ax.text(
            x1, y1 - 5, f"{label} {score:.2f}",
            color="white", fontsize=10, fontweight="bold",
            bbox=dict(facecolor=color, alpha=0.7, pad=2, edgecolor="none")
        )

        ys, xs = np.where(mask > 0)
        if len(xs) > 0:
            cx, cy = xs.mean(), ys.mean()
            ax.plot(cx, cy, marker="+", markersize=15, markeredgewidth=2.5, color="white")
            ax.plot(cx, cy, marker="+", markersize=15, markeredgewidth=1, color=color)

    ax.axis("off")
    ax.set_title(f'{os.path.basename(IMAGE_PATH)} — "{TEXT_PROMPT}"', fontsize=13)
    plt.tight_layout()

    # Save with the original filename (e.g. photo1_seg.jpg)
    stem    = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
    out_path = os.path.join(OUTPUT_FOLDER, f"{stem}_seg.jpg")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()   # ← important: prevents memory buildup across many images
    print(f"  Saved → {out_path}")

print(f"\n✅ Done. Results in {OUTPUT_FOLDER}")

replace /content/novel/__MACOSX/._novel? [y]es, [n]o, [A]ll, [N]one, [r]ename: N
Found 144 images in /content/novel/novel

[1/144] 000000_novel00.jpg
  Detected 1 object(s): ['hands']


/tmp/ipykernel_1476/3943850924.py:57: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  COLORS = plt.cm.get_cmap("tab10").colors


  Saved → /content/seg_results/000000_novel00_seg.jpg

[2/144] 000000_novel01.jpg
  Detected 1 object(s): ['hands']


/tmp/ipykernel_1476/3943850924.py:57: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  COLORS = plt.cm.get_cmap("tab10").colors


  Saved → /content/seg_results/000000_novel01_seg.jpg

[3/144] 000000_novel02.jpg
  Detected 1 object(s): ['hands']


...

  Saved → /content/seg_results/000105_novel03_seg.jpg

[131/144] 000105_novel04.jpg
  Detected 1 object(s): ['hands']


/tmp/ipykernel_1476/3943850924.py:57: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  COLORS = plt.cm.get_cmap("tab10").colors


  Saved → /content/seg_results/000105_novel04_seg.jpg

[132/144] 000105_novel05.jpg
  Detected 1 object(s): ['hands']


/tmp/ipykernel_1476/3943850924.py:57: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  COLORS = plt.cm.get_cmap("tab10").colors


  Saved → /content/seg_results/000105_novel05_seg.jpg

[133/144] 000110_novel00.jpg
  Detected 1 object(s): ['hands']
